In [ ]:
%load_ext autoreload
%autoreload 2

from __future__ import absolute_import, division, print_function
import torch
from trainer_endoda3 import Trainer
from options_endoda3 import MonodepthOptions


In [ ]:
# Minimal options for testing
options = MonodepthOptions()
args = [
    '--batch_size', '2',
    '--num_workers', '1',
    '--of_samples',
    '--of_samples_num', '10',
    '--frame_ids', '0', '-1', '1',
    '--dataset', 'endovis',
    '--data_path', '/mnt/cluster/workspaces/jinjingxu/SCARED_Images_Resized/',
    '--log_dir', '/tmp/endoda_debug',
    '--log_dir', '/mnt/cluster/workspaces/jinjingxu/tmp/',
    '--compute_metrics',
    '--depth_model_type', 'depthanything3',
    '--endoda3_model_config', '/mnt/cluster/workspaces/jinjingxu/proj/PAL-SfmLearner/networks/configs/endo-da3-all-wowrapper.yaml',
    '--pose_model_type', 'da3_internal',
    '--k_model_type', 'da3_internal',
    '--enable_seq_inputs',
    '--of_supervised_with_which', 'inputs_color',
    '--use_perframe_gt_K',
    '--learn_intrinsics',
    # '--of_model_type', 'raft',
    # '--use_raft_multi_iters',
    # '--raft_trainable_modules', 'convnormrelu layer1 layer2_0 ',
    # '--learn_intrinsics',
 
    # '--endoda3_model_config', '/mnt/cluster/workspaces/jinjingxu/proj/PAL-SfmLearner/networks/configs/endo-da3-depth-wowrapper.yaml'
]
opts = options.parse_notebook(args)

# Initialize trainer
trainer = Trainer(opts)
print(f"Trainer initialized on {trainer.device}")


In [ ]:
# Get sample batch
trainer.step = 0
trainer.set_train()
train_iter = iter(trainer.train_loader)
inputs = next(train_iter)

# Forward pass
outputs, losses = trainer.process_batch(inputs)

print("Forward pass completed!")
print(f"Output keys: {len(outputs)} keys")
print(f"Loss: {losses['loss'].item():.6f}")
print(f"Loss components: {list(losses.keys())}")


In [ ]:
# Compute losses explicitly
losses = trainer.compute_losses(inputs, outputs)

print("Loss computation:")
for key, val in losses.items():
    if isinstance(val, torch.Tensor):
        print(f"  {key}: {val.item():.6f}")
    else:
        print(f"  {key}: {val}")

# compute losses_0 explicitly
losses_0 = trainer.compute_losses_0(inputs, outputs)

print("Loss computation_0:")
for key, val in losses_0.items():
    if isinstance(val, torch.Tensor):
        print(f"  {key}: {val.item():.6f}")
    else:
        print(f"  {key}: {val}")

In [ ]:
# Get validation batch (has GT depth and poses)
trainer.set_eval()
val_iter = iter(trainer.val_loader)
val_inputs = next(val_iter)

# Forward pass
with torch.no_grad():
    val_outputs, val_losses = trainer.process_batch(val_inputs)

# Compute depth metrics
from utils.metrics import compute_depth_metrics, compute_pose_metrics

depth_metrics = compute_depth_metrics(val_inputs, val_outputs)
pose_metrics = compute_pose_metrics(val_inputs, val_outputs, opts.frame_ids)

print("Depth Metrics:")
if depth_metrics:
    for key, val in depth_metrics.items():
        print(f"  {key}: {val:.6f}")
else:
    print("  No depth metrics (GT depth not available)")

print("\nPose Metrics:")
if pose_metrics:
    for key, val in pose_metrics.items():
        print(f"  {key}: {val:.6f}")
else:
    print("  No pose metrics (GT poses not available)")


In [ ]:
# Full training step
trainer.set_train()
train_iter = iter(trainer.train_loader)
inputs = next(train_iter)

# Forward
outputs, losses = trainer.process_batch(inputs)

# Backward
trainer.model_optimizer.zero_grad()
losses["loss"].backward()
trainer.model_optimizer.step()

print(f"Training step completed! Loss: {losses['loss'].item():.6f}")
